# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import pandas as pd
from pathlib import Path

# Find the CSV regardless of where the notebook is run from in the repo
candidates = [
    'data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    'content_refresh_anonymized.csv',
]
csv_path = next((p for p in candidates if Path(p).exists()), None)
assert csv_path is not None, "CSV not found — check data/raw/ in your repo and update `candidates` above"
print('Loading:', csv_path)

df = pd.read_csv(csv_path)
print('Row count:', len(df))
print('Unique content_id:', df['content_id'].nunique())
print('Unique client_id:', df['client_id'].nunique())

# Grain check: does content_id uniquely identify a row?
dupes = df.groupby('content_id').size()
dupes = dupes[dupes > 1]
print('content_ids appearing more than once:', len(dupes))

# Confirm there is no date/report_date column (single snapshot, not a daily panel)
date_like_cols = [c for c in df.columns if 'date' in c.lower()]
print('Columns with "date" in the name:', date_like_cols)

# Sanity-check the two named windows are internally consistent
# (last_30d + prev_30d should each be <= the 90d total for the same metric)
window_check = (
    (df['impressions_last_30d'] + df['impressions_prev_30d']) <= df['impressions_90d']
).mean()
print(f'Share of rows where last30d + prev30d <= 90d total (impressions): {window_check:.3f}')


Loading: ../../data/raw/content_refresh_anonymized.csv


Row count: 30000
Unique content_id: 30000
Unique client_id: 32
content_ids appearing more than once: 0
Columns with "date" in the name: ['days_since_last_update']
Share of rows where last30d + prev30d <= 90d total (impressions): 1.000


**Contract claim (unit of analysis + time window):**

- One row = one pseudonymized content item (`content_id`), belonging to exactly one of 32 clients (`client_id`).
- There is **no `report_date` / daily grain** in this file — each row is a single snapshot summarizing that item's **trailing 90-day** window (e.g. `impressions_90d`, `clicks_90d`, `sessions_90d`), plus a nested **last-30-day vs. previous-30-day** comparison (`impressions_last_30d` / `impressions_prev_30d`, etc.) inside that same 90-day period.
- This is the *starter* dataset (`content_refresh_anonymized.csv`), not the warehouse panel — so "time window" here means "how far back the aggregated metrics reach as of one snapshot," not a range of calendar dates in the table. There is therefore no `month=2026-03` slice to take and no `IS TRUE` availability flag like the warehouse's `ga4_data_available` — the closest analogue in this file is `avg_position == 0` meaning "no data," which is verified in Section 3.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy** — the thing being predicted, or what it's computed from:
- `trend_direction` — categorical decline/growth label (down / up / stable / new / flat)
- `trend_pct` — the continuous number `trend_direction` is derived from
- `impressions_last_30d`, `impressions_prev_30d` — **verified below**: `trend_pct` is computed almost exactly from the percent change between these two columns. They are the label's raw inputs, not features.
- `clicks_last_30d`, `sessions_last_30d` — same 'last 30 days' window that defines the label period; excluded from features alongside impressions for consistency, even though the exact trend formula only uses impressions.

**Feature** — knowable before the moment of prediction, safe to use:
- `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`
- `word_count`, `char_count`
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`
- `days_with_impressions`, `days_with_sessions`
- `clicks_prev_30d`, `sessions_prev_30d` — history from *before* the label window, temporally safe (unlike their `_last_30d` counterparts)
- `content_age_days`, `days_since_last_update`
- `ctr`, `avg_position` (note: `avg_position == 0` means *no data*, not rank zero — 1,205 rows, must not be treated as a real position), `engagement_rate`
- `scroll_rate`, `ai_traffic_pct` (note: both can exceed 100 because numerator/denominator come from different measurement systems — not an error, per data dictionary)

**Context** — for grouping/joining/reading, never fed to a model:
- `content_id`, `client_id` — pseudonymous IDs, grouping and train/test-splitting only
- `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier` — human-readable bins of continuous fields already listed as features above; kept for reading/grouping, not included as separate model inputs to avoid double-counting the same signal in two forms

**Excluded** — each with a one-line why:
- `provider_used` — 71.5% missing; reflects which internal generation workflow/tool was used, an operational/product decision, not a search-performance signal
- `model_used` — 19.1% missing, same reasoning as `provider_used`: internal tooling metadata, not a search signal

In [2]:
# --- Verify every column is bucketed exactly once ---
buckets = {
    'label': ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d',
              'clicks_last_30d', 'sessions_last_30d'],
    'feature': ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent',
                'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
                'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
                'days_with_impressions', 'days_with_sessions', 'clicks_prev_30d', 'sessions_prev_30d',
                'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
                'scroll_rate', 'ai_traffic_pct'],
    'context': ['content_id', 'client_id', 'age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier',
                'char_count_tier', 'impression_tier', 'position_tier'],
    'excluded': ['provider_used', 'model_used'],
}
all_bucketed = sorted(sum(buckets.values(), []))
all_columns = sorted(df.columns.tolist())
print('Columns in dataframe but not bucketed:', set(all_columns) - set(all_bucketed))
print('Columns bucketed but not in dataframe:', set(all_bucketed) - set(all_columns))
print('Any column bucketed twice?', len(all_bucketed) != len(set(all_bucketed)))
print('Total columns:', len(all_columns), '| Total bucketed:', len(all_bucketed))

# --- Verify the label-leakage claim: trend_pct is derived from impressions_last_30d vs impressions_prev_30d ---
sub = df.dropna(subset=['trend_pct'])
implied_pct_change = (
    (sub['impressions_last_30d'] - sub['impressions_prev_30d'])
    / sub['impressions_prev_30d'].replace(0, pd.NA) * 100
)
corr = implied_pct_change.corr(sub['trend_pct'])
print(f'\nCorrelation(trend_pct, implied % change from impressions_last_30d vs prev_30d): {corr:.3f}')

# --- Verify avg_position == 0 gotcha (no data, not rank zero) ---
print('Rows with avg_position == 0 ("no data" flag, not a real rank):', (df['avg_position'] == 0).sum())

# --- Verify scroll_rate / ai_traffic_pct can exceed 100 ---
print('Rows with scroll_rate > 100:', (df['scroll_rate'] > 100).sum())
print('Rows with ai_traffic_pct > 100:', (df['ai_traffic_pct'] > 100).sum())

# --- Verify provider_used / model_used missingness (why they're excluded) ---
print('\nprovider_used missing %:', round(df['provider_used'].isnull().mean() * 100, 1))
print('model_used missing %:', round(df['model_used'].isnull().mean() * 100, 1))


Columns in dataframe but not bucketed: set()
Columns bucketed but not in dataframe: set()
Any column bucketed twice? False
Total columns: 44 | Total bucketed: 44

Correlation(trend_pct, implied % change from impressions_last_30d vs prev_30d): 1.000
Rows with avg_position == 0 ("no data" flag, not a real rank): 1205
Rows with scroll_rate > 100: 119
Rows with ai_traffic_pct > 100: 23

provider_used missing %: 71.5
model_used missing %: 19.1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**A note on scope:** the assignment card's phrasing (`month=2026-03`, `IS TRUE` availability flags) describes the Hugging Face warehouse panel (`fact_content_daily_performance_sample`, etc). This repo's data folder only ships the starter snapshot (`content_refresh_anonymized.csv`) — there is no HF token configured here, so the three verification queries below are the same three checks (grain, scope/count, availability) run against the file actually on disk, adapted to a single-snapshot table instead of a daily panel. That substitution is itself logged as a limitation in Section 4.

### Query 1 — Grain: one row really is one `content_id`

In [3]:
# Grain probe: group by the claimed unit and look for any group with more than one row
grain_probe = df.groupby('content_id').size()
violations = grain_probe[grain_probe > 1]
print('content_id groups with count > 1:', len(violations))
print('Grain holds:', len(violations) == 0)


content_id groups with count > 1: 0
Grain holds: True


### Query 2 — Scope: row count, client count, and how rows are spread across clients

In [4]:
rows_per_client = df.groupby('client_id').size()
print('Total rows:', len(df))
print('Total clients:', df['client_id'].nunique())
print('Rows per client — min/median/mean/max:',
      rows_per_client.min(), rows_per_client.median(), round(rows_per_client.mean(), 1), rows_per_client.max())

# Window-alignment check: does last_30d + prev_30d roughly reconstruct the 90d total, per metric?
for metric in ['impressions', 'clicks', 'sessions']:
    share_ok = (
        (df[f'{metric}_last_30d'] + df[f'{metric}_prev_30d']) <= df[f'{metric}_90d']
    ).mean()
    print(f'{metric}: share of rows where last30d + prev30d <= 90d total = {share_ok:.3f}')


Total rows: 30000
Total clients: 32
Rows per client — min/median/mean/max: 3 567.0 937.5 7008
impressions: share of rows where last30d + prev30d <= 90d total = 1.000
clicks: share of rows where last30d + prev30d <= 90d total = 1.000
sessions: share of rows where last30d + prev30d <= 90d total = 1.000


### Query 3 — Availability: missingness is patterned, not random (this file's stand-in for `IS TRUE` filtering)

In [5]:
# Overall missingness for the columns most likely to bite a model
cols_to_check = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc',
                  'main_intent', 'provider_used', 'model_used', 'trend_pct']
print('Overall missing %:')
print((df[cols_to_check].isnull().mean() * 100).round(1))

print('\nMissingness by content_type (the pattern behind the overall numbers):')
print('word_count missing %, by content_type:')
print((df.groupby('content_type')['word_count'].apply(lambda s: s.isnull().mean()) * 100).round(1))
print('\nsearch_volume missing %, by content_type:')
print((df.groupby('content_type')['search_volume'].apply(lambda s: s.isnull().mean()) * 100).round(1))

# The "no data" flag: avg_position == 0, and whether that too is patterned by content_type
print('\navg_position == 0 ("no data", not rank zero) — overall count:', (df['avg_position'] == 0).sum(),
      f"({(df['avg_position'] == 0).mean()*100:.1f}% of rows)")
print('avg_position == 0 rate, by content_type:')
print((df.groupby('content_type')['avg_position'].apply(lambda s: (s == 0).mean()) * 100).round(1))


Overall missing %:
word_count       25.7
char_count       25.7
search_volume     8.2
competition       8.2
cpc               8.2
main_intent       7.9
provider_used    71.5
model_used       19.1
trend_pct        11.3
dtype: float64

Missingness by content_type (the pattern behind the overall numbers):
word_count missing %, by content_type:


content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

search_volume missing %, by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4
Name: search_volume, dtype: float64

avg_position == 0 ("no data", not rank zero) — overall count: 1205 (4.0% of rows)
avg_position == 0 rate, by content_type:
content_type
comparison article     0.0
feedly article        34.8
keyword article        1.7
Name: avg_position, dtype: float64


**Findings:** the grain holds (0 duplicate `content_id`s across 30,000 rows). The 32 clients carry very unbalanced row counts (as few as 3, as many as ~7,000 rows), so any split must group by `client_id` rather than sample rows independently. `word_count`/`char_count` are missing almost entirely for one `content_type` and barely at all for the others; `search_volume` is missing for essentially all `feedly article` rows and almost none of the rest. Both confirm the skill's warning: missingness tracks `content_type`, so a blind `fillna(0)` would silently teach the model "this is a feedly article" instead of "this has no keyword data." `avg_position == 0` behaves the same way — it's concentrated in `feedly article` rows, not spread evenly, so it needs a `has_position` flag rather than being read as rank 0.

### Five features, max — with an "available when?" line for each

Built from the same file/slice already verified above (this dataset has no month partitions to choose between).

In [6]:
feature_cols = ['search_volume', 'competition', 'cpc', 'content_age_days', 'days_since_last_update']
feature_frame = df[['content_id', 'client_id'] + feature_cols].copy()
print(feature_frame.shape)
feature_frame.head()


(30000, 7)


,content_id,client_id,search_volume,competition,cpc,content_age_days,days_since_last_update
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,187,20
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,445,25
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,141,20
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,0.00,463,22
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,0.00,263,14


- `search_volume` — knowable at the decision moment because it's a keyword-research property of the topic, set before the content was ever written.
- `competition` — knowable at the decision moment because it's the same pre-publication keyword-research metric as `search_volume`.
- `cpc` — knowable at the decision moment because it's an external ad-market price for the target keyword, independent of how this specific page performs.
- `content_age_days` — knowable at the decision moment because it's simply "today minus publish date," no future data required.
- `days_since_last_update` — knowable at the decision moment because it only counts backward from today to the last edit, never forward.

### The trap — add one label-derived column on purpose, watch the score jump, then remove it

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Reuse the label framing from ML-03: is_declining_label = 1 when trend_direction == "down"
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

honest_features = ['search_volume', 'competition', 'cpc', 'content_age_days', 'days_since_last_update']

# --- Honest model: features only, no label-window columns ---
work = df.dropna(subset=honest_features + ['is_declining_label']).copy()
X, y = work[honest_features], work['is_declining_label']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
print('Honest AUC (5 features only):', round(honest_auc, 3))

# --- The trap: sneak trend_pct in as a "feature" ---
# trend_pct is literally what trend_direction is thresholded from — it should never be a feature.
leaky_features = honest_features + ['trend_pct']
work_leak = df.dropna(subset=leaky_features + ['is_declining_label']).copy()
Xl, yl = work_leak[leaky_features], work_leak['is_declining_label']
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
leaked_auc = roc_auc_score(yte_l, LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l).predict_proba(Xte_l)[:, 1])
print('Leaked AUC (5 features + trend_pct):', round(leaked_auc, 3))

print(f"\nJump: {honest_auc:.3f} -> {leaked_auc:.3f} just from adding the column the label is computed from.")

# --- Remove the leak, keep the honest number ---
leaky_features_removed = [c for c in leaky_features if c != 'trend_pct']
assert leaky_features_removed == honest_features
print('\nLeak removed. Feature set is back to:', leaky_features_removed)
print('Honest number to report going forward: AUC =', round(honest_auc, 3))


Honest AUC (5 features only): 0.599


Leaked AUC (5 features + trend_pct): 1.0

Jump: 0.599 -> 1.000 just from adding the column the label is computed from.

Leak removed. Feature set is back to: ['search_volume', 'competition', 'cpc', 'content_age_days', 'days_since_last_update']
Honest number to report going forward: AUC = 0.599


**The lesson, confirmed on real data:** the honest 5-feature model barely beats a coin flip (AUC ≈ 0.60) — these pre-publication signals carry only weak decline signal on their own. Adding `trend_pct`, a column the label is directly computed from, pushes AUC to a near-perfect 1.0. That's not a better model, it's the label leaking into the features. The number that goes in any writeup from here on is the honest 0.60, not the 1.0.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# Supporting numbers for the limitations written below
print('Clients:', df['client_id'].nunique(), '| rows per client min/max:',
      df.groupby('client_id').size().min(), '/', df.groupby('client_id').size().max())
print('provider_used missing %:', round(df['provider_used'].isnull().mean() * 100, 1))
print('model_used missing %:', round(df['model_used'].isnull().mean() * 100, 1))
print('trend_pct missing %:', round(df['trend_pct'].isnull().mean() * 100, 1),
      '(these rows have no usable label at all)')
print('content_type counts:')
print(df['content_type'].value_counts())


Clients: 32 | rows per client min/max: 3 / 7008
provider_used missing %: 71.5
model_used missing %: 19.1
trend_pct missing %: 11.3 (these rows have no usable label at all)
content_type counts:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


**Named limitations of this slice:**

1. **Single snapshot, no daily panel.** This file has no `report_date` — it's one trailing-90-day summary per content item, not a time series. It can support "is this item currently declining," but it cannot show *when* a decline started, how fast it's moving, or support the per-client `gsc_data_start` / `ga4_data_available` history checks the warehouse tables allow.
2. **Content-type-driven missingness, not random.** `search_volume`/`competition`/`cpc` are essentially absent for `feedly article` rows, and `word_count`/`char_count` are ~28% missing specifically for `keyword article` rows. Any model trained across all `content_type`s must add `has_*` availability flags rather than impute a default — the missingness itself is close to a `content_type` label in disguise.
3. **Unbalanced client history.** Rows per client range from 3 to ~7,000. A handful of large clients can dominate a pooled model; splits must group by `client_id`, and any per-client metric should be read with its sample size attached.
4. **11.3% of rows have no `trend_pct` at all** — these items have no usable label and must be dropped from label-dependent work, not silently imputed to 'stable'.
5. **This is a single gated extract, not the warehouse.** There's no way, from this file alone, to confirm whether a given client's 90-day window is fully backed by real GSC/GA4 history or partly zero-filled before that client's own data start — the kind of check the warehouse's `ga4_data_available` flag exists for. Anything computed here should be read as *directional*, not as ground truth for any one client's true trajectory.

## Self-check

Before you submit, confirm each line honestly:

- [☑] Every section above is filled — markdown thinking AND the code that backs it
- [☑] The notebook runs top to bottom with no errors (Runtime → Run all)
- [☑] No client names, URLs, or private queries anywhere
- [☑] My claims use careful words: observed, measured, directional, decision-support
- [☑] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.